## initialize

In [ ]:
# init
from importlib import reload
import os
from pathlib import Path
import pandas as pd
from IPython.display import clear_output
import boto3
import sys
import time

import toolsGeneral.main as tgm
import toolsGeneral.logger as tgl
import toolsOSM.overpass as too
import toolsPandas.helpers as tph
import toolsSync.main as tsm

def pckgs_reload():
    reload(tgm)
    reload(too)
    reload(tph)
    reload(tgl)
    reload(tsm)

ROOT = Path.cwd().parents[0]
DATA_DIR = ROOT / "data"
TESTS_DIR = DATA_DIR / 'tests results'
CLEANED_DIR = DATA_DIR / 'cleaned'
DEV_MODE = True

task = 'test_first_level'
TEST_FIRST_LEVEL_DIR = TESTS_DIR / 'osm first level test'
process_state = tgm.load(DATA_DIR / "process_state.json")
first_level_test_state = tgm.load(DATA_DIR / "first_level_test_state.json")
first_level_test_to_delete = tgm.load(DATA_DIR  / "first_level_test_to_delete.json")
print(len(process_state))

logger = tgl.initiate_logger('logger', TEST_FIRST_LEVEL_DIR / 'first_level_test.log')

214
214


## osm test if first level divisions belong to country

### * select countries to test

In [6]:
countries_tested = [c for c, val in process_state.items() if (val[task]['status'] == 'ok')]
logger.info(f"countries tested: {len(countries_tested)}")
countries_to_test = [c for c, val in process_state.items() if 
    (val['clean']['status'] == 'ok') and (val[task]['status'] in ['pending', 'error'])
]
logger.info(f"countries to test: {len(countries_to_test)}")

[INFO] : countries tested: 60
[INFO] : countries to test: 23


In [7]:
countries_to_test = ['Armenia']

### * initialize B2

In [58]:
session = boto3.session.Session()

s3 = session.client(
    service_name="s3",
    aws_access_key_id=os.environ["B2_KEY_ID"],
    aws_secret_access_key=os.environ["B2_APPLICATION_KEY"],
    endpoint_url=os.environ["B2_ENDPOINT"]
)

### * download required data

In [4]:
logger.info(f"* Downloading required data to test: {len(countries_to_test)} countries")
logger.info(f"  * Downloading data from B2 in directory: '{CLEANED_DIR.relative_to(ROOT)}'")

countries_downloaded = tsm.donwload_country_data_from_bucket(countries_to_test, os.environ["B2_BUCKET_NAME"], CLEANED_DIR.relative_to(ROOT), CLEANED_DIR, s3, logger)

logger.info(f'* Countries to test: {len(countries_to_test)}')
logger.info(f"* Countries to test with downloaded cleaned data from B2: {len(countries_downloaded)}")

[INFO] : * Downloading required data to test: 1 countries
[INFO] :   * Downloading data from B2 in directory: 'data\cleaned'
[INFO] :   * Countries to get data: 1
[INFO] :     * Found in B2: 1, Missing in B2: 0
[INFO] :   * Downloading data for countries: 1
[INFO] :     * Directory: 'data\cleaned' -> 'c:\Users\gonta\D\study\full stack\projects\administrative divisions osm scrape\data\cleaned'
[INFO] :     * (1/1) Country Armenia files found: 1
[INFO] :       * Skip existing file Armenia_cleaned.pkl
[INFO] :   * Number of downloaded files: 1/1
[INFO] : * Countries to test: 1
[INFO] : * Countries to test with downloaded cleaned data from B2: 1


### * load data for countries to test

In [8]:
logger.info(f"* Load data from: {CLEANED_DIR.relative_to(ROOT)}")
cleaned_files = [f for f in CLEANED_DIR.glob('*/*')]
logger.info(f"  * Directories found: {len(cleaned_files)}")

# to_test_df = {file.parent.name:tgm.load(str(file)) for file in cleaned_files if file.parent.name}
countries_to_test_df = {file.parent.name:tgm.load(str(file)) for file in cleaned_files if file.parent.name in countries_to_test}
logger.info(f"  * Countries to test: {len(countries_to_test)}")
logger.info(f"  * Data loaded for countries to test: {len(countries_to_test_df)}")

[INFO] : * Load data from: data\cleaned
[INFO] :   * Directories found: 83
[INFO] :   * Countries to test: 1
[INFO] :   * Data loaded for countries to test: 1


In [9]:
first_lvl_df = {}
countries_wihout_first_level = []
for country, df in countries_to_test_df.items():
    if not df[df['tags.admin_level'] == '4'].empty:
        first_lvl_df[country] = df[df['tags.admin_level'] == '4']
    else:
        countries_wihout_first_level.append(country)
logger.info(f"countries with first level: {len(first_lvl_df)}")
logger.info(f"countries without first level: {countries_wihout_first_level}")

[INFO] : countries with first level: 1
[INFO] : countries without first level: []


### * filter already processed relations

In [ ]:
logger.info(f"first_level_test_state: {len(first_level_test_state)}")
logger.info(f"tuples processed: {sum([len(t['processed']) for t in first_level_test_state.values()])}")

[INFO] : first_level_test_state: 61
[INFO] : tuples processed: 1211


In [11]:
first_lvl_filtered_df = {}
for country,df in first_lvl_df.items():
    processed = first_level_test_state[country]['processed'] if first_level_test_state.get(country) else set()
    failed = first_level_test_state[country]['failed'] if first_level_test_state.get(country) else set()

    in_processed = df[['id','tags.parent_id','tags.country_id']].apply(tuple, axis=1).isin(processed)
    in_failed = df[['id','tags.parent_id','tags.country_id']].apply(tuple, axis=1).isin(failed)
    filtered_df = df[~in_processed | in_failed]
    if not filtered_df.empty:
        first_lvl_filtered_df[country] = filtered_df

logger.info(f"Countries with first level -> filtered pending to process: {len(first_lvl_filtered_df)}")
logger.info(f"Total of pending to process: {sum([len(lis) for lis in first_lvl_filtered_df.values()])}")

[INFO] : Countries with first level -> filtered pending to process: 0
[INFO] : Total of pending to process: 0


### * make tests

In [ ]:
for country, df in first_lvl_filtered_df.items():
    if not first_level_test_state.get(country):
        first_level_test_state[country] = {"to_process": set(), "processed": set(), "failed": set(), "next_index": 0}
    country_test_state = first_level_test_state[country]

    total = len(df)
    test_res = {}
    save_path = TEST_FIRST_LEVEL_DIR / country / f"{country}_first_level_test_res_{country_test_state['next_index']}.pkl"
    country_test_state['next_index'] += 1

    for i, (idx, row) in enumerate(df.iterrows(), start=1):
        id_triplet = (row["id"], row["tags.parent_id"], row["tags.country_id"])
        logger.info(f" ^ [{i}/{total}]: testing {id_triplet}:")

        res = too.is_child_inside_parent(row["id"], row["tags.parent_id"])
        test_res[id_triplet] = res

        status_list = [v['status'] for k,v in res.items()]
        if 'error' in status_list:
            country_test_state['failed'].add(id_triplet)
        else:
            country_test_state['processed'].add(id_triplet)
            country_test_state['failed'].remove(id_triplet)
            
        logger.info(f"  * saving ...")
        tgm.dump(save_path, test_res)
        tgm.dump(DATA_DIR / "first_level_test_state.json", first_level_test_state)

        resume  = {k:v['status'] for k,v in res.items()}
        logger.info(f" $ finished: status: {resume}")
        
        time.sleep(2)
        clear_output(wait=True)

[INFO] :  ^ [1/1]: testing ('364080', '364066', '364066'):
[INFO] :    > Getting admin_centre:
[INFO] :     * Found node (admin_centre)
[INFO] :    > Testing admin_centre node (id: 6123773911)
[INFO] :    * Attempt 1 failed: http_error: 504 Server Error: Gateway Timeout for url: http://overpass-api.de/api/interpreter?data=%0A++++++++%5Bout%3Ajson%5D%5Btimeout%3A300%5D%3B%0A++++++++rel%28364066%29%3B%0A++++++++map_to_area%3B%0A++++++++node%286123773911%29-%3E.testnode%3B%0A++++++++node.testnode%28area%29%3B%0A++++++++out+ids%3B%0A++++
[INFO] :    * Attempt 2 failed: http_error: 504 Server Error: Gateway Timeout for url: http://overpass-api.de/api/interpreter?data=%0A++++++++%5Bout%3Ajson%5D%5Btimeout%3A300%5D%3B%0A++++++++rel%28364066%29%3B%0A++++++++map_to_area%3B%0A++++++++node%286123773911%29-%3E.testnode%3B%0A++++++++node.testnode%28area%29%3B%0A++++++++out+ids%3B%0A++++
[INFO] :    * Attempt 3 failed: http_error: 504 Server Error: Gateway Timeout for url: http://overpass-api.de/api

### * load test res data

In [36]:
# test_res_by_cntr = {f.parent.name:tgm.load(f) for f in TEST_FIRST_LEVEL_DIR.glob('*/*') if f.parent.name in countries_to_test}
# test_res_by_cntr = {f.parent.name:tgm.load(f) for f in TEST_FIRST_LEVEL_DIR.glob('*/*') if f.parent.name}
test_res_by_cntr = tgm.load_countries_dirs(TEST_FIRST_LEVEL_DIR, countries_to_test)
logger.info(f'Results per country found: {len(test_res_by_cntr)}')

test_res_by_tuple = {k:v for list in test_res_by_cntr.values() for k,v in list.items()}
logger.info(f'Results per element found: {len(test_res_by_tuple)}')

[INFO] : Results per country found: 1
[INFO] : Results per element found: 11


### * review

In [37]:
result_status = [[[k,v['status']] for k,v in ele.items()] for ele in test_res_by_tuple.values()]
pd.Series(list(result_status)).value_counts()

[[admin_centre, ok], [label, missing], [place, ok], [geom_node, ok], [centroid, ok]]    10
[[admin_centre, ok], [label, ok], [place, ok], [geom_node, ok], [centroid, ok]]          1
Name: count, dtype: int64

In [38]:
pd.Series([ele[1] for lis in result_status for ele in lis]).value_counts()

ok         45
missing    10
Name: count, dtype: int64

In [39]:
results = [[[k,v['result']] if v['status'] == 'ok' else  [k,v['status']] for k,v in ele.items() ] for ele in test_res_by_tuple.values()]
pd.Series(results).value_counts()

[[admin_centre, True], [label, missing], [place, True], [geom_node, True], [centroid, True]]    10
[[admin_centre, True], [label, True], [place, True], [geom_node, True], [centroid, True]]        1
Name: count, dtype: int64

### * select false entities to delete

In [43]:
MIN_TESTS = 2
KEEP_THRESHOLD = 0.3
miss = 0
test_res_by_tuple_bool = {}
for id, val in test_res_by_tuple.items():
    true_count = 0
    false_count = 0
    for type in ['admin_centre', 'label', 'place', 'geom_node', 'centroid']:
        if val[type]['status'] == 'ok':
            # make a voting weight selection
            if val[type]['result'] is True:
                true_count += 1
            else:
                false_count += 1
    if (true_count + false_count) < MIN_TESTS:
        test_res_by_tuple_bool[id] = 'missing'
    else:
        true_ratio = true_count / ((false_count + true_count))
        # test_res_by_tuple_bool[id] = true_count >= false_count
        test_res_by_tuple_bool[id] = true_ratio > KEEP_THRESHOLD


logger.info(f"Data types: {tgm.tally([v for k,v in test_res_by_tuple_bool.items()])}")

[INFO] : Data types: Counter({True: 11})


In [44]:
relations_from_test_to_delete = {k for k,v in test_res_by_tuple_bool.items() if v is False}
logger.info(f'Relations from test to delete: {len(relations_from_test_to_delete)}')

[INFO] : Relations from test to delete: 0


### * From the relations to delete, select the childs in the country that has them as parent

In [45]:
id_triplets = pd.concat(countries_to_test_df.values(), ignore_index=True)[['id', 'tags.parent_id', 'tags.country_id']].fillna('missing').apply(tuple,axis=1).to_list()
logger.info(f"All dataframes id triplets: {len(id_triplets)}")

[INFO] : All dataframes id triplets: 27


In [46]:
parents_to_delete = relations_from_test_to_delete
relations_childs_to_delete = set()
while len(parents_to_delete) > 0:
    parents_id_and_countryid = {(ele[0],ele[2]) for ele in parents_to_delete}
    childs_to_delete = {ele for ele in id_triplets if (ele[1], ele[2]) in parents_id_and_countryid}
    relations_childs_to_delete.update(childs_to_delete)
    parents_to_delete = childs_to_delete
logger.info(f"Childs to delete: {len(relations_childs_to_delete)}")

[INFO] : Childs to delete: 0


In [78]:
logger.info(f"Old total of relations to delete from first level test: {len(first_level_test_to_delete)}")

[INFO] : Old total of relations to delete from first level test: 4932


In [80]:
relations_to_delete = relations_from_test_to_delete | relations_childs_to_delete
logger.info(f"Current relations to delete: {len(relations_to_delete)}")

[INFO] : Current relations to delete: 0


In [ ]:
first_level_test_to_delete.update(relations_to_delete)
logger.info(f"New total of relations to delete from first level test: {len(first_level_test_to_delete)}")

[INFO] : New total of relations to delete from first level test: 4932


In [72]:
tgm.dump(DATA_DIR  / "first_level_test_to_delete.json", first_level_test_to_delete)

### * upload results to B2

In [ ]:
logger.info("* Uploading data to backblaze b2")
config = {'root':ROOT, 's3':s3, 'logger':logger}
task = 'test_first_level'
tested_dirs = [dir.name for dir in TEST_FIRST_LEVEL_DIR.glob('*/') if dir.name in countries_to_test]

if len(tested_dirs) < 1:
    logger.info("No data to upload, exiting script")
    sys.exit(0)
else:
    logger.info(f"* Test result directories found: {len(tested_dirs)}")

for country in tested_dirs:
    process_status = 'ok'
    process_error = None
    
    # all data in country directory will  be uploaded
    if not DEV_MODE:
        upload_response = tsm.upload_dir_files_to_backblaze(TEST_FIRST_LEVEL_DIR / country, config)
        process_status = upload_response['status']
        process_error = upload_response['status_type']

    # override process task state with upload response
    logger.info(f"  * Updating {country} in process state: ({task}, ok)")
    tsm.update_process_state(process_state, country, task, process_status=process_status, process_error=process_error)
    tgm.dump(DATA_DIR / "process_state.json", process_state)

[INFO] : * Uploading data to backblaze b2
[INFO] : * Test result directories found: 1
[INFO] : Uploading directory: c:\Users\gonta\D\study\full stack\projects\administrative divisions osm scrape\data\tests results\osm first level test\Armenia
[INFO] : Number of files found: 2
[INFO] : Uploaded Armenia_first_level_test_res_0.pkl to Backblaze successfully
[INFO] : Uploaded Armenia_first_level_test_res_1.pkl to Backblaze successfully
[INFO] :   * Updating Armenia in process state: (test_first_level, ok)


### commit files

In [ ]:
if not DEV_MODE:
    tsm.commit_file(DATA_DIR / "process_state.json", f"Update process state for {country}: ({task}, ok)", logger)
    tsm.commit_file(DATA_DIR  / "first_level_test_to_delete.json", f"Update {task} to delete relations, current length {len(first_level_test_to_delete)}", logger)
    tsm.commit_file(DATA_DIR  / "first_level_test_state.json", f"Update {task} state: {len(first_level_test_state)}", logger)